# The Essence of Attention and Transformers
### Step-by-Step Demo: Following the Transformer Data Flow
Ref: Attention Is All You Need https://arxiv.org/abs/1706.03762

```
Text --> [1. Tokenize] --> [2. Embed] --> [3. + Position] --> [4. Attention] --> [5. Multi-Head] --> [6. Encoder] --> [7. Decoder]
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)

---
## Step 1: Sequence and Tokenization

A tokenizer splits text into tokens (words or subwords) and maps each one
to a unique integer ID using a vocabulary lookup table.

In [ ]:
# 1a. Build a simple vocabulary

# A vocabulary is a mapping from words to integer IDs.
# Real tokenizers (like BPE or WordPiece) handle subwords

vocab = {
    '<PAD>': 0,    # padding token (used to fill short sequences)
    '<UNK>': 1,    # unknown token (for words not in vocab)
    'i'    : 2,
    'love' : 3,
    'ai'   : 4,
    'the'  : 5,
    'cat'  : 6,
    'sat'  : 7,
    'on'   : 8,
    'mat'  : 9,
}

vocab_size = len(vocab)
print(f'Vocabulary size: {vocab_size} words')
print(f'Vocabulary: {vocab}')

In [ ]:
# 1b. Tokenize a sentence

def tokenize(sentence, vocab):
    words = sentence.lower().split()
    return [vocab.get(w, vocab['<UNK>']) for w in words]

sentence = 'the cat sat on the mat'
token_ids = tokenize(sentence, vocab)

print(f'Input sentence: "{sentence}"')
print(f'Token IDs:      {token_ids}')
print()

words = sentence.lower().split()
for word, tid in zip(words, token_ids):
    print(f'  "{word}" --> {tid}')

print()
print(f'Sequence length: {len(token_ids)}')

---
## Step 2: Word Embedding


```
token ID 6 ('cat') --> [0.12, -0.45, 0.78, ...]   (d_model numbers)
token ID 7 ('sat') --> [0.33,  0.21, -0.55, ...]   (d_model numbers)
```

The embedding is simply a lookup table (a matrix of shape vocab_size x d_model).
Row i of the table is the vector for token ID i.
These vectors are learned during training.

In [ ]:
# 2a. Create an embedding table

d_model = 8   # embedding dimension (small for readability; paper uses 512)

# The embedding table: one row per word in the vocabulary.
# In a real model these values are learned. Here we use random initialization.
embedding_table = np.random.randn(vocab_size, d_model)

print(f'Embedding table shape: {embedding_table.shape}  (vocab_size x d_model)')
print(f'  {vocab_size} words, each represented by a {d_model}-dimensional vector.')
print()
print('Embedding table (each row is one word vector):')
print(np.round(embedding_table, 3))

In [ ]:
# 2b. Look up embeddings for our token sequence

# Embedding is just indexing into the table: take row [token_id]
token_ids_array = np.array(token_ids)
embedded_sequence = embedding_table[token_ids_array]   # shape: (seq_len, d_model)

print(f'Token IDs:  {token_ids}')
print(f'Words:      {words}')
print(f'Embedded sequence shape: {embedded_sequence.shape}  (seq_len x d_model)')
print()

for i, (word, tid) in enumerate(zip(words, token_ids)):
    vec = np.round(embedded_sequence[i], 3)
    print(f'  "{word}" (id={tid}) --> {vec}')

In [ ]:
# 2c. Visualize: same word = same vector

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(embedded_sequence, aspect='auto', cmap='RdYlBu')

ax.set_yticks(range(len(words)))
ax.set_yticklabels([f'{w} (id={t})' for w, t in zip(words, token_ids)])
ax.set_xlabel('Embedding Dimension')
ax.set_ylabel('Token in Sequence')
ax.set_title('Embedded Sequence\n'
             'Same word ("the") has identical rows')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## Step 3: Positional Encoding

Formula:
```
PE[pos, 2i]   = sin( pos / 10000^(2i / d_model) )
PE[pos, 2i+1] = cos( pos / 10000^(2i / d_model) )
```

In [ ]:
# 3a. Build the Positional Encoding matrix

def positional_encoding(max_len, d_model):
    PE  = np.zeros((max_len, d_model))
    for k in range(max_len):
        # Used for mapping to column indices 0 <= i < d_model/2, with a single value of i maps to both sine and cosine functions
        for i in np.arange(int(d_model/2)):
            denominator = np.power(10000, 2*i/d_model)
            PE[k, 2*i] = np.sin(k/denominator)
            PE[k, 2*i+1] = np.cos(k/denominator)
    return PE

# 50 and 128 servs for the demo to be easy for visual
MAX_LEN = 50
D_MODEL = 128

PE = positional_encoding(MAX_LEN, D_MODEL)

print('Positional Encoding shape:', PE.shape)
print(f'  {MAX_LEN} positions, each with a unique {D_MODEL}-dimensional fingerprint.')
print()
print('First 3 positions, first 8 dimensions:')
print(np.round(PE[:3, :8], 4))

In [ ]:
# 3b. Heatmap of the full PE matrix

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(PE, aspect='auto', cmap='RdYlBu', origin='upper')

ax.xaxis.set_label_position('top')
ax.xaxis.tick_top()
ax.set_xlabel('Embedding Dimension')

ax.set_ylabel('Position in Sequence')
ax.set_title('Positional Encoding Heatmap\nEach row is one position fingerprint', pad=20)

cbar = plt.colorbar(im, ax=ax)
cbar.ax.set_xlabel('value', labelpad=8)
cbar.ax.xaxis.set_label_position('bottom')

plt.tight_layout()
plt.show()

print('Left side (low dims): high frequency, captures fine position differences.')
print('Right side (high dims): low frequency, captures coarse position differences.')

In [ ]:
# 3d. Show how PE is added to embeddings

# Reuse the embedding from Step 2 (small example for readability)
sample_words = ['the', 'cat', 'sat', 'on', 'the', 'mat']
sample_seq_len = len(sample_words)

sample_embeddings = embedded_sequence
sample_pe = positional_encoding(sample_seq_len, d_model)

combined = sample_embeddings + sample_pe

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

titles = ['Token Embeddings\n(WHAT the word is)',
          'Positional Encoding\n(WHERE the word is)',
          'Combined = Embedding + PE\n(WHAT + WHERE)']
data   = [sample_embeddings, sample_pe, combined]

for ax, title, matrix in zip(axes, titles, data):
    im = ax.imshow(matrix, aspect='auto', cmap='RdYlBu')
    ax.set_yticks(range(sample_seq_len))
    ax.set_yticklabels(sample_words)
    ax.set_xlabel('Dimension')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.show()

print('Left: same word "the" has identical rows (positions 0 and 4).')
print('Middle: positions 0 and 4 have different PE vectors.')
print('Right: after adding PE, the two "the" tokens are now distinguishable.')

---
## Step 4: Attention Mechanism (NumPy)

Now we apply attention to the embedded + positionally-encoded tokens from Steps 2-3.

In self-attention, the same input is used as Query, Key, and Value.
In practice, Q, K, V are created by passing the input through separate linear projections
(learned weight matrices). Here we simulate this with random projections.

The formula:
```
Attention(Q, K, V) = softmax( Q @ K.T / sqrt(d_k) ) @ V
```

- Q (Query) => what each token is asking for
- K (Key) => what each token offers as a label
- V (Value) => the actual content to retrieve

The dot product Q @ K.T measures similarity between every pair of tokens.
Softmax converts scores into weights that sum to 1.0 per row.
The output is a weighted blend of the value vectors.

In [ ]:
# 4a. Set up Q, K, V for self-attention

# Reuse the sentence and embeddings from Steps 1-3
words = sentence.lower().split()   # ['the', 'cat', 'sat', 'on', 'the', 'mat']
seq_len = len(words)               # 6
d_k = combined.shape[1]            # 8 (same as d_model from Step 2)

# In SELF-ATTENTION, Q, K, V all come from the same input.
# In a trained model, each goes through a separate learned linear projection
# (Q = input @ W_Q, etc.). Those projections are introduced in Step 5.
# Here we use the raw input directly to focus on the attention mechanism itself.

Q = combined   # shape: (seq_len, d_k) = (6, 8)
K = combined
V = combined

print(f'Sentence: "{sentence}"')
print(f'seq_len = {seq_len}, d_k = {d_k}')
print(f'Q shape: {Q.shape}   K shape: {K.shape}   V shape: {V.shape}')
print()
print('Q = K = V = combined embeddings from Step 3 (self-attention).')

In [ ]:
# 4b. Compute raw similarity scores: Q @ K.T

# scores[i, j] = how relevant token j is to token i
scores = Q @ K.T   # shape: (seq_len, seq_len) = (6, 6)

print(f'scores = Q @ K.T,  shape: {scores.shape}')
print()
print(np.round(scores, 2))
print()
print('Each cell [i,j] is the raw attention score from token i to token j.')

In [ ]:
# 4c. Scale by sqrt(d_k)

# Without scaling, large d_k causes dot products to grow large.
# Softmax on large values produces near one-hot output, killing gradients.
# Dividing by sqrt(d_k) keeps variance close to 1.

scaled_scores = scores / np.sqrt(d_k)

print(f'Scale factor: 1 / sqrt({d_k}) = {1/np.sqrt(d_k):.4f}')
print()
print('Scaled scores:')
print(np.round(scaled_scores, 3))

In [ ]:
# 4d. Apply Softmax row-wise

def softmax(x):
    # Subtract row max before exp for numerical stability
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

attention_weights = softmax(scaled_scores)   # shape: (6, 6)

# Note: some values appear very small because with random (untrained)
# embeddings, certain tokens happen to be much more similar than others.
# Softmax amplifies these differences. In a trained model, learned
# projections (W_Q, W_K, W_V) distribute attention more meaningfully.
print("Attention weights (after softmax):")
print(np.round(attention_weights, 4))
print()
print("Row sums (must all equal 1.0):")
print(np.round(attention_weights.sum(axis=-1), 6))

In [ ]:
# 4e. Weighted sum: attention_weights @ V

output = attention_weights @ V   # shape: (seq_len, d_k) = (6, 8)
print("Weighted sum:")
print(np.round(output, 4))

In [ ]:
# 4f. Visualize attention weights

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(attention_weights, cmap="Blues", vmin=0, vmax=1)

ax.set_xticks(range(seq_len))
ax.set_xticklabels(words)
ax.set_yticks(range(seq_len))
ax.set_yticklabels(words)
ax.set_xlabel("Key  (token being attended to)")
ax.set_ylabel("Query  (token doing the attending)")
ax.set_title("Attention Weights for: "" + sentence + ""\n"
             "Each row is a probability distribution (sums to 1.0)")

for i in range(seq_len):
    for j in range(seq_len):
        val = attention_weights[i, j]
        text_color = "white" if val > 0.5 else "black"
        # Show 3 decimals so small values are visible
        ax.text(j, i, f"{val:.3f}",
                ha="center", va="center", fontsize=8, color=text_color)

plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

---
## Step 5: Multi-Head Attention (Keras)

In Step 4 we used a single attention computation.
The problem: one attention can only capture one type of relationship at a time.

Multi-Head Attention runs h separate attentions in parallel,
each with its own learned projections (W_Q, W_K, W_V).
Each head can learn a different relationship (e.g., one head for syntax,
another for coreference, another for position).

```
MultiHead(Q, K, V) = Concat(head_1, ..., head_h) @ W_O
head_i             = Attention(Q @ W_Qi,  K @ W_Ki,  V @ W_Vi)
```

Tensor shape journey:
```
(batch, seq, d_model)           => input
  -> linear projections
(batch, seq, d_model)
  -> split into h heads
(batch, h, seq, d_k)            => d_k = d_model / h
  -> scaled dot-product attention per head
(batch, h, seq, d_k)
  -> concat heads + final projection
(batch, seq, d_model)           => same shape as input
```

In [ ]:
# 5a. Scaled Dot-Product Attention as a Keras layer
# This is the Keras version of what we built in Step 4 with NumPy.
class ScaledDotProductAttention(layers.Layer):
    def call(self, Q, K, V, mask=None):
        d_k = tf.cast(tf.shape(K)[-1], tf.float32)
        scores = tf.matmul(Q, K, transpose_b=True)
        scaled = scores / tf.math.sqrt(d_k)

        if mask is not None:
            scaled += mask * -1e9

        weights = tf.nn.softmax(scaled, axis=-1)
        output  = tf.matmul(weights, V)
        return output, weights

In [ ]:
# 5b. Multi-Head Attention layer

class MultiHeadAttention(layers.Layer):
    def __init__(self, d_model, num_heads, **kwargs):
        super().__init__(**kwargs)
        assert d_model % num_heads == 0, 'd_model must be divisible by num_heads'

        self.d_model   = d_model
        self.num_heads = num_heads
        self.d_k       = d_model // num_heads

        # These are the learned projections that Step 4 was missing.
        # Each head gets its own slice of these projections.
        self.W_Q = layers.Dense(d_model, use_bias=False, name='W_Q')
        self.W_K = layers.Dense(d_model, use_bias=False, name='W_K')
        self.W_V = layers.Dense(d_model, use_bias=False, name='W_V')
        self.W_O = layers.Dense(d_model, use_bias=False, name='W_O')
        self.sdpa = ScaledDotProductAttention()

    def split_into_heads(self, x, batch_size):
        # (batch, seq, d_model) -> (batch, heads, seq, d_k)
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.d_k))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, Q, K, V, mask=None):
        batch_size = tf.shape(Q)[0]

        # Project through learned weight matrices
        Q = self.W_Q(Q)
        K = self.W_K(K)
        V = self.W_V(V)

        # Split d_model dimension across heads
        Q = self.split_into_heads(Q, batch_size)
        K = self.split_into_heads(K, batch_size)
        V = self.split_into_heads(V, batch_size)

        # Attention per head
        attn_out, weights = self.sdpa(Q, K, V, mask)

        # Merge heads: (batch, heads, seq, d_k) -> (batch, seq, d_model)
        attn_out = tf.transpose(attn_out, perm=[0, 2, 1, 3])
        attn_out = tf.reshape(attn_out, (batch_size, -1, self.d_model))

        # Final linear projection
        output = self.W_O(attn_out)
        return output, weights

In [ ]:
# 5c. Run multi-head attention and trace tensor shapes

D_MODEL = 64    # embedding dimension (paper uses 512, small here for speed)
H = 4           # number of heads (paper uses 8)
SEQ_LEN = 6     # same as our sentence length
BATCH = 1       # single example

# Simulated input: one sentence of 6 tokens, each a 64-dim vector
x = tf.random.normal((BATCH, SEQ_LEN, D_MODEL))

mha = MultiHeadAttention(D_MODEL, H)
out, attn_weights = mha(x, x, x)   # self-attention: Q = K = V

print(f'Config: d_model={D_MODEL}, num_heads={H}, d_k={D_MODEL//H}')
print()
print('Shape journey:')
print(f'  Input:             {x.shape}         (batch, seq, d_model)')
print(f'  After split heads: ({BATCH}, {H}, {SEQ_LEN}, {D_MODEL//H})    (batch, heads, seq, d_k)')
print(f'  After attention:   ({BATCH}, {H}, {SEQ_LEN}, {D_MODEL//H})    (batch, heads, seq, d_k)')
print(f'  After concat:      ({BATCH}, {SEQ_LEN}, {D_MODEL})       (batch, seq, d_model)')
print(f'  Output:            {out.shape}         (batch, seq, d_model) -- same as input!')
print(f'  Weights:           {attn_weights.shape}  (batch, heads, seq, seq)')

In [ ]:
# 5d. Visualize: each head learns a different attention pattern

w = attn_weights[0].numpy()   # shape: (heads, seq, seq) -- first sample
labels = words if len(words) == SEQ_LEN else [f't{i}' for i in range(SEQ_LEN)]

fig, axes = plt.subplots(1, H, figsize=(4 * H, 4))
if H == 1:
    axes = [axes]

for h_idx, ax in enumerate(axes):
    im = ax.imshow(w[h_idx], cmap='Blues', vmin=0, vmax=w.max())
    ax.set_xticks(range(SEQ_LEN))
    ax.set_xticklabels(labels, fontsize=8, rotation=45)
    ax.set_yticks(range(SEQ_LEN))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_title(f'Head {h_idx + 1}')
    if h_idx == 0:
        ax.set_ylabel('Query')
    ax.set_xlabel('Key')

fig.suptitle('Multi-Head Attention: Each Head Learns a Different Pattern\n'
             f'Same input, {H} parallel views of the sequence',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

print(f'Each head has its own W_Q, W_K, W_V projections.')
print(f'So each head "sees" the sequence differently.')
print(f'The outputs from all {H} heads are concatenated and projected by W_O.')

---
## Step 6: Transformer Encoder (Keras)

The Encoder Layer combines everything from Steps 3-5 into one block.
It has two sub-layers, each with a skip connection and layer normalization:

```
Input (batch, seq, d_model)
  |                                                         
  +---> Multi-Head Self-Attention ---> Dropout --+           
  |                                             |           
  +---------------------------------------------+--> Add & LayerNorm  (skip connection 1)
                                                      |               
  +---> Feed-Forward Network -------> Dropout --+     |               
  |                                             |     |               
  +---------------------------------------------+--> Add & LayerNorm  (skip connection 2)
                                                      |
                                                   Output (batch, seq, d_model)
```

Key concepts:
- Skip connection: out = LayerNorm(x + sublayer(x)). Adds input directly to output,
  creating a gradient highway that prevents vanishing gradients in deep networks.
- FFN: two dense layers with ReLU. FFN(x) = ReLU(x @ W1 + b1) @ W2 + b2.
  Typically d_ff = 4 * d_model.
- A full encoder stacks N identical layers (paper uses N=6).

In [ ]:
# 6a. Feed-Forward Network

class FeedForwardNetwork(layers.Layer):
    def __init__(self, d_model, d_ff, **kwargs):
        super().__init__(**kwargs)
        self.dense1 = layers.Dense(d_ff,    activation='relu', name='FFN_expand')
        self.dense2 = layers.Dense(d_model,                    name='FFN_project')

    def call(self, x):
        # Expand to d_ff dimensions, then project back to d_model
        return self.dense2(self.dense1(x))

In [ ]:
# 6b. Encoder Layer

class EncoderLayer(layers.Layer):
    def __init__(self, d_model, num_heads, d_ff, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.mha     = MultiHeadAttention(d_model, num_heads)
        self.ffn     = FeedForwardNetwork(d_model, d_ff)
        self.norm1   = layers.LayerNormalization(epsilon=1e-6)
        self.norm2   = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(dropout_rate)
        self.dropout2 = layers.Dropout(dropout_rate)

    def call(self, x, training=False, mask=None):
        # Sub-layer 1: Multi-Head Self-Attention + skip connection
        attn_out, attn_weights = self.mha(x, x, x, mask)
        attn_out = self.dropout1(attn_out, training=training)
        out1 = self.norm1(x + attn_out)   # skip connection: add input back

        # Sub-layer 2: Feed-Forward Network + skip connection
        ffn_out = self.ffn(out1)
        ffn_out = self.dropout2(ffn_out, training=training)
        out2 = self.norm2(out1 + ffn_out) # skip connection: add out1 back

        return out2, attn_weights

In [ ]:
# 6c. Full Transformer Encoder (N layers + embedding + PE)

class TransformerEncoder(tf.keras.Model):
    def __init__(self, num_layers, d_model, num_heads, d_ff,
                 vocab_size, max_seq_len, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.d_model   = d_model
        self.embedding = layers.Embedding(vocab_size, d_model)

        # Positional encoding (same formula from Step 3, as a constant)
        self.pos_enc = positional_encoding(max_seq_len, d_model).astype(np.float32)

        self.enc_layers = [
            EncoderLayer(d_model, num_heads, d_ff, dropout_rate, name=f'enc_layer_{i}')
            for i in range(num_layers)
        ]
        self.dropout = layers.Dropout(dropout_rate)

    def call(self, token_ids, training=False, mask=None):
        seq_len = tf.shape(token_ids)[1]

        # Step 2: Embedding
        x = self.embedding(token_ids)
        x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))  # scale as in paper

        # Step 3: Add positional encoding
        x += self.pos_enc[:seq_len, :]
        x = self.dropout(x, training=training)

        # Pass through N encoder layers
        all_weights = []
        for enc_layer in self.enc_layers:
            x, w = enc_layer(x, training=training, mask=mask)
            all_weights.append(w)

        return x, all_weights

print('TransformerEncoder model defined.')
print('It combines: Embedding + PE + N x EncoderLayer.')

In [ ]:
# 6d. Build and run the encoder, trace tensor shapes

# Parameters (scaled down for demo; paper uses d_model=512, heads=8, N=6)
NUM_LAYERS = 2
D_MODEL    = 64
NUM_HEADS  = 4
D_FF       = 256    # = 4 * d_model
VOCAB_SIZE = len(vocab)  # from Step 1
MAX_SEQ    = 50

encoder = TransformerEncoder(
    num_layers=NUM_LAYERS, d_model=D_MODEL, num_heads=NUM_HEADS,
    d_ff=D_FF, vocab_size=VOCAB_SIZE, max_seq_len=MAX_SEQ
)

# Use our tokenized sentence from Step 1
input_ids = tf.constant([token_ids])  # add batch dimension: (1, 6)

enc_output, enc_weights = encoder(input_ids, training=False)

print(f'Encoder config: {NUM_LAYERS} layers, d_model={D_MODEL}, heads={NUM_HEADS}, d_ff={D_FF}')
print()
print(f'Input token IDs:  {input_ids.shape}  (batch, seq_len)')
print(f'Encoder output:   {enc_output.shape}  (batch, seq_len, d_model)')
print(f'Attention weights: {len(enc_weights)} layers, each {enc_weights[0].shape}')
print()
print('Input shape (batch, seq_len) --> Output shape (batch, seq_len, d_model).')
print('Each token now has a context-aware representation of d_model dimensions.')

In [ ]:
# 6e. Model summary

encoder.summary()

In [ ]:
# 6g. Visualize attention from each encoder layer

fig, axes = plt.subplots(NUM_LAYERS, NUM_HEADS, figsize=(4 * NUM_HEADS, 4 * NUM_LAYERS))
if NUM_LAYERS == 1:
    axes = [axes]

for layer_idx in range(NUM_LAYERS):
    w = enc_weights[layer_idx][0].numpy()  # (heads, seq, seq)
    for h_idx in range(NUM_HEADS):
        ax = axes[layer_idx][h_idx]
        im = ax.imshow(w[h_idx], cmap='Blues', vmin=0)
        ax.set_xticks(range(len(words)))
        ax.set_xticklabels(words, fontsize=7, rotation=45)
        ax.set_yticks(range(len(words)))
        ax.set_yticklabels(words, fontsize=7)
        ax.set_title(f'Layer {layer_idx+1}, Head {h_idx+1}', fontsize=9)

fig.suptitle('Attention Patterns Across Encoder Layers and Heads\n'
             'Different layers and heads capture different relationships',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()